# ReCTS recogniser: retrain, export, **and keep it**

Reviewer item 2 needs stock YOLOv11 and the paper's detector read by *the same*
fine-tuned recogniser. That recogniser no longer exists: it was exported to
`/kaggle/working/PP-OCRv5_server_rec_infer` but the archiving line in the original
notebook was commented out, so it went when the session was cleared.

This notebook rebuilds it and saves it. Nothing to attach — the data comes from Drive.

**Accelerator: GPU T4 x2. Runtime ~3 h.** Use *Save Version -> Save & Run All*, then
add this notebook's output as an input to the control notebook.

In [ ]:
PADDLE_ROOT = "/kaggle/working/PaddleOCR"
DATA_ROOT = "/kaggle/working/content/train_data/rec"
CONFIG = PADDLE_ROOT + "/configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml"
DICTIONARY = PADDLE_ROOT + "/configs/rec/multi_language/custom_reCTS_dict.txt"
PRETRAINED = PADDLE_ROOT + "/pretrained_models/PP-OCRv5_mobile_rec_pretrained.pdparams"
EXPORT_DIR = "/kaggle/working/PP-OCRv5_rects_rec_infer"

EPOCHS = 35          # as published
BATCH_SIZE = 64
GPUS = "0,1"

In [ ]:
!pip install -q gdown
!git clone -q --branch ablation-mscbam-probe https://github.com/SaiSanthosh1508/End-to-End-Text-Translation-Pipeline.git /kaggle/working/repo
import sys; sys.path.insert(0, "/kaggle/working/repo")

## 1. ReCTS training images and line annotations

In [ ]:
!gdown -q 1orMtLhJt3rQl3pMoLm31eh-SmDG74W1K -O /kaggle/working/ReCTS.zip
!unzip -q -o /kaggle/working/ReCTS.zip -d /kaggle/working
!ls /kaggle/working | head

In [ ]:
from pathlib import Path
from rects_control.crops import build

train_n, val_n = build(Path("/kaggle/working/img"), Path("/kaggle/working/gt"), Path(DATA_ROOT))
print(f"{train_n} train crops, {val_n} val crops")
assert train_n > 10_000, "far fewer crops than expected - check the unzipped layout"

## 2. PaddleOCR and the PP-OCRv5 mobile checkpoint

In [ ]:
!git clone -q https://github.com/PaddlePaddle/PaddleOCR.git {PADDLE_ROOT}
!pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install -q -r {PADDLE_ROOT}/requirements.txt
!pip install -q lmdb rapidfuzz
!wget -q -P {PADDLE_ROOT}/pretrained_models https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams

In [ ]:
from rects_control.paddle_config import build_dictionary, patch_training_config

n_chars = build_dictionary(Path(DICTIONARY))
patch_training_config(
    Path(CONFIG), data_root=Path(DATA_ROOT), dictionary=Path(DICTIONARY),
    pretrained=Path(PRETRAINED), epochs=EPOCHS, batch_size=BATCH_SIZE,
)
print(f"{n_chars} characters in the label space")

## 3. Fine-tune

~2 h on T4 x2. The eval accuracy printed at the end is the sanity check: PP-OCRv5
mobile fine-tuned on ReCTS crops should land well above 0.7 on the held-out 10%.

In [ ]:
!python3 -m paddle.distributed.launch --gpus '{GPUS}' {PADDLE_ROOT}/tools/train.py \
    -c {CONFIG} -o Global.pretrained_model={PRETRAINED}

## 4. Export for inference

In [ ]:
!python3 {PADDLE_ROOT}/tools/export_model.py -c {CONFIG} -o \
    Global.pretrained_model={PADDLE_ROOT}/output/PP-OCRv5_mobile_rec/best_model/model.pdparams \
    Global.save_inference_dir={EXPORT_DIR}
!ls -la {EXPORT_DIR}

### Does it actually read ReCTS text?

A broken export — wrong dictionary, wrong checkpoint — still loads and still returns
strings. Reading held-out crops whose ground truth we know is the only cheap way to
tell, and it costs seconds here versus discovering it after the 4.4 h detector run.

In [ ]:
import cv2
from rects_control.recognizer import PaddleRecognizer

rec = PaddleRecognizer(Path(EXPORT_DIR))
rows = Path(DATA_ROOT, "rec_gt_test.txt").read_text(encoding="utf-8").splitlines()[:12]
paths, truth = zip(*(r.split("\t") for r in rows))

predicted = rec([cv2.imread(str(Path(DATA_ROOT, p))) for p in paths])
hits = sum(p == t for p, t in zip(predicted, truth))
for p, t in zip(predicted, truth):
    print(f"{'ok ' if p == t else '   '} pred={p!r:20s} gt={t!r}")
print(f"\n{hits}/{len(truth)} exact on a 12-crop sample")
assert hits >= 4, "export looks wrong - do not spend GPU hours on the control yet"

## 5. Persist it

The step whose omission cost the original weights. `/kaggle/working` survives only
if you *Save Version*, so archive first and save after.

In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/rects_rec_finetuned", "zip", EXPORT_DIR)
print(archive, Path(archive).stat().st_size // 1024, "KB")
assert Path(archive).stat().st_size > 1_000_000, "archive is suspiciously small"

**Now: Save Version -> Save & Run All.**

When it finishes, open the control notebook and add this notebook's output under
*Add Input -> Notebook Output*. Also worth doing once: publish
`/kaggle/working/PP-OCRv5_rects_rec_infer` as a Kaggle dataset, so a cleared session
can never cost you these weights again.